In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, f1_score, recall_score, fbeta_score, average_precision_score
from sklearn.model_selection import train_test_split
from Utils.Preprocess import load_data, preprocess

In [2]:
# Load and preprocess
train_df, test_df = load_data()
X, y = preprocess(train_df, mode="baseline")

# Identify categorical columns
categorical_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(f"[INFO] Categorical columns: {categorical_cols}")

# Train-Test-Val Split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)
print(f"[INFO] Train shape: {X_train.shape}")
print(f"[INFO] Validation shape: {X_val.shape}")
print(f"[INFO] Test shape: {X_test.shape}")

[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)
[INFO] Categorical columns: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']
[INFO] Train shape: (413378, 395)
[INFO] Validation shape: (88581, 395)
[INFO] Test shape: (88581, 395)


In [3]:
# scale-pos weight
n_fraud = y_train.sum()
n_nonfraud = len(y_train) - n_fraud
scale_pos_weight = n_nonfraud / n_fraud
print(f"[INFO] scale_pos_weight: {scale_pos_weight:.2f}")

# CatBoost parameters
catboost_params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 6,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "AUC",
    "loss_function": "Logloss",
    "random_seed": 42,
    "early_stopping_rounds": 50,
    "verbose": 100
}

# Train model
train_pool = Pool(X_train, y_train, cat_features=categorical_cols)
val_pool = Pool(X_val, y_val, cat_features=categorical_cols)

model = CatBoostClassifier(**catboost_params)
model.fit(train_pool, eval_set=val_pool, use_best_model=True)

# Get ROC-AUC and PR-AUC
y_prob = model.predict_proba(X_val)[:,1]
roc_auc = roc_auc_score(y_val, y_prob)
pr_auc = average_precision_score(y_val, y_prob)
print(f"Validation ROC-AUC: {roc_auc:.4f}")
print(f"Validation PR-AUC: {pr_auc:.4f}")

[INFO] scale_pos_weight: 27.58
0:	test: 0.8088714	best: 0.8088714 (0)	total: 533ms	remaining: 8m 52s
100:	test: 0.8890277	best: 0.8890277 (100)	total: 38.3s	remaining: 5m 40s
200:	test: 0.9004995	best: 0.9004995 (200)	total: 1m 14s	remaining: 4m 54s
300:	test: 0.9116866	best: 0.9116866 (300)	total: 1m 50s	remaining: 4m 16s
400:	test: 0.9205743	best: 0.9205743 (400)	total: 2m 28s	remaining: 3m 42s
500:	test: 0.9268489	best: 0.9268489 (500)	total: 3m 6s	remaining: 3m 6s
600:	test: 0.9310657	best: 0.9310657 (600)	total: 3m 43s	remaining: 2m 28s
700:	test: 0.9345279	best: 0.9345309 (699)	total: 4m 20s	remaining: 1m 51s
800:	test: 0.9375130	best: 0.9375130 (800)	total: 4m 58s	remaining: 1m 14s
900:	test: 0.9399781	best: 0.9399781 (900)	total: 5m 37s	remaining: 37.1s
999:	test: 0.9419679	best: 0.9419679 (999)	total: 6m 16s	remaining: 0us

bestTest = 0.941967885
bestIteration = 999

Validation ROC-AUC: 0.9420
Validation PR-AUC: 0.6547


In [4]:
# Best threshold on val set (maximise F2)
val_preds = model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.01, 0.99, 0.01)
best_f2 = -1
best_thresh = 0.5
for t in thresholds:
    f2 = fbeta_score(y_val, (val_preds > t).astype(int), beta=2)
    if f2 > best_f2:
        best_f2 = f2
        best_thresh = t

print(f"[INFO] Best threshold on validation set (F2): {best_thresh:.2f}, F2: {best_f2:.4f}")

[INFO] Best threshold on validation set (F2): 0.69, F2: 0.6404


In [5]:
# Evaluate on test set
test_preds = model.predict_proba(X_test)[:, 1]
test_labels = (test_preds > best_thresh).astype(int)

auc = roc_auc_score(y_test, test_preds)
f1 = f1_score(y_test, test_labels)
recall = recall_score(y_test, test_labels)
f2 = fbeta_score(y_test, test_labels, beta=2)

print("\n[INFO] Test Metrics:")
print(f"ROC-AUC: {auc:.6f}")
print(f"F1 Score: {f1:.4f}")
print(f"F2 Score: {f2:.4f}")
print(f"Recall: {recall:.4f}")


[INFO] Test Metrics:
ROC-AUC: 0.946977
F1 Score: 0.5702
F2 Score: 0.6565
Recall: 0.7302


In [6]:
# Train final model on all training data (train + val)
X_full_train = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full_train = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)
full_pool = Pool(X_full_train, y_full_train, cat_features=categorical_cols)

final_model = CatBoostClassifier(**catboost_params)
final_model.fit(full_pool)

0:	total: 423ms	remaining: 7m 2s
100:	total: 42.9s	remaining: 6m 21s
200:	total: 1m 24s	remaining: 5m 37s
300:	total: 2m 6s	remaining: 4m 53s
400:	total: 2m 48s	remaining: 4m 11s
500:	total: 3m 29s	remaining: 3m 28s
600:	total: 4m 10s	remaining: 2m 46s
700:	total: 4m 52s	remaining: 2m 4s
800:	total: 5m 35s	remaining: 1m 23s
900:	total: 6m 17s	remaining: 41.4s
999:	total: 6m 58s	remaining: 0us


CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, scale_pos_weight=np.float64(27.5797842920354), verbose=100)

In [7]:
# Evaluate final model on test set
final_test_preds = final_model.predict_proba(X_test)[:, 1]
final_test_labels = (final_test_preds > best_thresh).astype(int)

final_auc = roc_auc_score(y_test, final_test_preds)
final_f1 = f1_score(y_test, final_test_labels)
final_recall = recall_score(y_test, final_test_labels)
final_f2 = fbeta_score(y_test, final_test_labels, beta=2)

print("\n[INFO] Final Model Test Metrics:")
print(f"ROC-AUC: {final_auc:.6f}")
print(f"F1 Score: {final_f1:.4f}")
print(f"F2 Score: {final_f2:.4f}")
print(f"Recall: {final_recall:.4f}")


[INFO] Final Model Test Metrics:
ROC-AUC: 0.948897
F1 Score: 0.5759
F2 Score: 0.6627
Recall: 0.7367


In [8]:
# Save final model
final_model.save_model("checkpoints/catboost_model.cbm")
print("[INFO] CatBoost model saved!")

X_test, _ = preprocess(test_df, mode="baseline")
test_pool = Pool(X_test, cat_features=categorical_cols)
test_preds = final_model.predict_proba(test_pool)[:, 1]
test_labels = (test_preds > best_thresh).astype(int)

[INFO] CatBoost model saved!
